# Check abundance ratios of plasma data

In the TurboID pipeline, tissue data is filtered using ROC on ratio to Cre- while plasma is not filtered

This notebook checks which plasma proteins would be filtered out with simililar Cre- ratio filters 

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import seaborn as sns
from pathlib import Path

# experimental results
data_dir = Path(
    "/Users/henrysanford/dev/test_data/turboid_intensity_filter"
)

experiment = "20221130_EV1_137"

protein_signal_intensity = pl.read_csv(
    data_dir / f"{experiment}.csv"
).select("uniprot_id","annotation", cs.contains("cre+_"), cs.contains("cre-_"))
ax = sns.boxplot(
    y="variable",
    x="value",
    data=protein_signal_intensity.unpivot(index=["uniprot_id", "annotation"]).sort(by = "variable"),
    log_scale=True,
    fliersize=1
)
ax.set_xlabel("signal intensity")
ax.set_ylabel(None)
ax.set_title(experiment)

plt.show()

Looking into filtering the plasma data based on FC from cre-, in the same way we filtered the tissue data. The ROC filtering we use for tissue data relies on False Positive (FP) proteins, defined as mitochondrial matrix proteins. Only two of our plasma experiments quantify any FP proteins: EV1_137 and CRW_B30. When we apply ROC filters to these experiments,

In [ ]:
aliases = pl.read_csv("/Users/henrysanford/dev/turboid-figures/reference_data/protein_aliases.csv", truncate_ragged_lines=True)

aliases  = dict(zip(aliases["protein"].to_list(), aliases["alias"].to_list()))

In [ ]:
import numpy as np
import math

roc_filtered_path = Path(
    "/Volumes/Users/hsanford/Documents/turbo_id_data/census_out_processing_results_plasma_only/results/7/05_results/01_tissue"
)
no_roc_path = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/03_results/04_turboid_downstream_results/05_results/02_serum"
)

metadata_df = pl.read_csv(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/02_metadata/conditions_metadata.csv"
).with_columns(
    pl.col("File name").str.split("census-out_").list.get(1).alias("experiment")
).filter(
    pl.col("file_type").eq("serum")
).select(["conditions","experiment"])
rows = []

fc_cutoffs = np.round(np.linspace(1.0, 3.2, int((3.2 - 1.0) / 0.05) + 1), 2)
for fc_cutoff in [1.15]:
    for experiment in metadata_df["experiment"]:
        exp_short = experiment.split("_",maxsplit=1)[1]
        print(exp_short)
        no_roc_df = pl.read_csv(
            no_roc_path / f"{experiment}/final_protein_table_{experiment}.csv"
        ).drop("").with_columns(pl.col("Entry Name").str.split("_").list.get(0).replace(aliases).alias("protein"))

        data = no_roc_df.select(
            cs.contains("(median_cre+/(median_cre-)"), "uniprot_id", "annotation","Entry Name","Mass"
        ).with_columns(
            pl.col("annotation").fill_null("other")
        ).unpivot(
            index=["uniprot_id", "annotation", "Entry Name", "Mass"]
        ).with_columns(
            pl.col("value").log(base = 2).alias("log2(cre+/cre-)"),
            pl.col("variable").str.split("_").list.get(0).alias("condition"),
            pl.col("Entry Name").str.split("_").list.get(0).replace(aliases).alias("protein"),
        )

        print(data.filter(pl.col("protein").eq("APOA4")))

        dropped_proteins = data.group_by(
            ["protein",]
        ).agg(
            pl.col("log2(cre+/cre-)").max()
        ).filter(pl.col("log2(cre+/cre-)") < math.log2(fc_cutoff))

        no_roc_df.filter(pl.col("protein").is_in(dropped_proteins["protein"]).not_()).write_csv(
            no_roc_path / f"{experiment}/final_protein_table_{experiment}_enrichment_filtered.csv"
        )
        
        rows.append({
            "experiment":exp_short,
            "proteins before FC cutoff filter": len(no_roc_df),
            "proteins after FC cutoff filter": len(no_roc_df) - len(dropped_proteins),
            "proteins filtered out": list(dropped_proteins["protein"]),
            "fc_cutoff": fc_cutoff,
            "percentage_dropped":(len(dropped_proteins) / len(no_roc_df)) * 100
        })
df = pl.from_dicts(rows)

    #df.write_csv(file = "/Users/henrysanford/dev/turboid-figures/fc_filter_summary.tsv", separator="\t")

In [ ]:
fc_requirement = 1.15
for exp in set(df["experiment"]):
    print(exp)
    f = list(df.filter(pl.col("fc_cutoff").eq(fc_requirement), pl.col("experiment").eq(exp))["proteins filtered out"])[0]
    print(", ".join(f))

sns.lineplot(data = df.drop("proteins filtered out"), x = "fc_cutoff", y = "percentage_dropped", hue = "experiment",palette="Set2" )
sns.set_style("whitegrid")
plt.ylabel("% of proteins dropped")
plt.xlabel("cre+/cre- FC requirement")
plt.legend(loc= "upper right")
plt.axvline(x=fc_requirement, color='red', linestyle='-')

In [ ]:

plt.figure(figsize=(4,1.5))
sns.set_style("whitegrid")
sns.boxplot(
    data = data,
    y = "condition",
    x = "log2(cre+/cre-)",
    fliersize=0,
    fill=None,
    color = "black"
)
sns.stripplot(
    data = data,
    y = "condition",
    x = "log2(cre+/cre-)",
    color = "grey",
    size = 4,
    alpha = 0.8,
)

plt.title(experiment)

plt.show()

# ER GSEA

Looking at enrichment of endoplasmic reticulum stress genes in TurboID data

In [ ]:
import gseapy as gp

# perform GSEA for ER stress on HFD data
pairs = [
    ["20240915_CRW_B25D", "eWAT_HFD vs. iWAT_HFD"],
    ["20240501_CRW_B17D", "eWAT_HFD vs. iWAT_HFD"],
    ["20240909_CRW_B25B", "eWAT_HFD vs. eWAT_chow"],
    ["20240429_CRW_B17B", "eWAT_HFD vs. eWAT_chow"],
    ["20240427_CRW_B17A", "iWAT_HFD vs. iWAT_chow"],
    ["20240907_CRW_B25A", "iWAT_HFD vs. iWAT_chow"],

]
metric = "log2_FC"
gsea_results = []
# read differential expression and filter to ER data
for pair in pairs:
    print(pair)
    hfd_df = (
        pl.read_csv(
            "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/03_results/04_turboid_downstream_results/05_results/turboid_differential_expression.csv"
        )
        .filter(
            pl.col("experiment").eq(pair[0]),
            pl.col("comparison").eq(pair[1])
        )
        .with_columns(
            pl.col("log2_FC").sign().mul(pl.col("-log10_pval")).alias("signed_p_value")
        )
        .select("p_value", "log2_FC", "-log10_pval", "gene_name", "signed_p_value")
    )

    rnk = hfd_df.select("log2_FC", "gene_name").to_pandas().set_index("gene_name").sort_values(by = "log2_FC")
    rnk = hfd_df.select(metric, "gene_name").to_pandas().set_index("gene_name").sort_values(by = metric)

    print(rnk.head())



    response_to_er_stress = {
        "GOBP_RESPONSE_TO_ENDOPLASMIC_RETICULUM_STRESS":["Abca7","Abl1","Aff4","Afg2b","Agr2","Aifm1","Akt1","Akt2","Akt3","Alox15","Alox5","Amfr","Anks4b","Ankzf1","Apaf1","App","Aqp11","Asb11","Atad3a","Atf3","Atf4","Atf6","Atf6b","Atg10","Atp2a1","Atp2a3","Atxn3","Aup1","Bag6","Bak1","Bax","Bbc3","Bbs1","Bbs10","Bcap31","Bcl2","Bcl2l1","Bcl2l11","Bfar","Bhlha15","Bid","Bok","Brsk2","Calr","Calr3","Calr4","Canx","Casp12","Casp3","Casp9","Cav1","Ccdc47","Ccnd1","Cdk5rap3","Cebpb","Cep290","Cert1","Cftr","Chac1","Clgn","Clu","Cops5","Creb3","Creb3l1","Creb3l2","Creb3l3","Crebrf","D1Pas1","Dab2ip","Ddit3","Ddrgk1","Ddx3x","Derl1","Derl2","Derl3","Dnajb12","Dnajb2","Dnajb9","Dnajc10","Dnajc3","Dnm1l","Ecpas","Edem1","Edem2","Edem3","Eif2a","Eif2ak2","Eif2ak3","Eif2ak4","Eif2b5","Eif2s1","Eif4g1","Elavl4","Erlec1","Erlin1","Erlin2","Ermp1","Ern1","Ern2","Ero1a","Erp27","Erp29","Erp44","Faf1","Faf2","Fbxo17","Fbxo2","Fbxo27","Fbxo44","Fbxo6","Fcgr2b","Ficd","Fis1","Flot1","Foxred2","Get4","Gorasp2","Grina","Gsk3b","H13","Herpud1","Herpud2","Hsp90b1","Hspa5","Hyou1","Ifng","Igtp","Ikbkg","Ins2","Itpr1","Jagn1","Jkamp","Jun","Kcnj8","Lhx1os","Lpcat3","Lrrk2","Man1a","Man1a2","Man1b1","Man1c1","Manf","Map3k20","Map3k5","Marchf6","Marcks","Mbtps2","Nccrp1","Nck1","Nck2","Nfe2l1","Nfe2l2","Nhlrc1","Niban1","Nod1","Nod2","Nploc4","Nr1h2","Nr1h3","Nrbf2","Nrros","Nupr1","Opa1","Os9","P4hb","Park7","Parp16","Parp6","Parp8","Pdia2","Pdia3","Pdia4","Pdia6","Pdx1","Pigbos1","Pik3r1","Pik3r2","Pla2g6","Pmaip1","Pml","Pmp22","Ppp1r15a","Ppp1r15b","Ppp2cb","Prkn","Psmc6","Ptpn1","Ptpn2","Qrich1","Rasgrf1","Rasgrf2","Rcn3","Rhbdd1","Rnf103","Rnf121","Rnf139","Rnf145","Rnf183","Rnf185","Rnf186","Rnf5","Rnft1","Rnft2","Rpap2","Scamp5","Sdf2l1","Sec16a","Sec61b","Sec61bl","Sel1l","Sel1l2","Selenok","Selenon","Selenos","Serinc3","Serp1","Serp2","Sesn2","Sgf29","Sgta","Sirt1","Spop","Srpx","Stc2","Stt3b","Stub1","Svip","Syvn1","Tardbp","Tbl2","Thbs1","Thbs4","Tmbim4","Tmbim6","Tmco1","Tmed2","Tmem117","Tmem129","Tmem238l","Tmem258","Tmem259","Tmem33","Tmem67","Tmtc3","Tmtc4","Tmub1","Tmub2","Tmx1","Tnfrsf10b","Tor1a","Trib3","Trim13","Trim25","Trp53","Txndc12","Uba5","Ubac2","Ube2g2","Ube2j1","Ube2j2","Ube2k","Ube4a","Ube4b","Ubqln1","Ubqln2","Ubxn1","Ubxn10","Ubxn2a","Ubxn4","Ubxn6","Ubxn8","Ufc1","Ufd1","Ufl1","Ufm1","Umod","Usp13","Usp14","Usp19","Usp25","Vapb","Vcp","Wfs1","Xbp1","Yod1"]
    }

    pre_res = gp.prerank(
        rnk = rnk,
        gene_sets = "/Users/henrysanford/Downloads/m5.go.bp.v2025.1.Mm.symbols.gmt",
        threads = 4,
        permutation_num=1000,
        outdir = None,
        verbose = True
    )
    ## easy way
    terms = pre_res.res2d.Term
    axs = pre_res.plot(terms=terms[0]) # v1.0.5
    plt.show()

    df = pre_res.res2d

    df["experiment"] = pair[0]
    df["comparison"] = pair[1]
    gsea_results.append(df)


In [ ]:
import polars as pl

df = pl.read_csv(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/03_results/04_turboid_downstream_results/05_results/turboid_differential_expression.csv"
)

aliases = pl.read_csv("/Users/henrysanford/dev/turboid-figures/reference_data/protein_aliases.csv", truncate_ragged_lines=True)
aliases  = dict(zip(aliases["protein"].to_list(), aliases["alias"].to_list()))

# First create a combined column for the pivot
df_with_combined = (
    df.drop("number_of_proteins")
    .fill_null("")
    .with_columns(
        pl.concat_str(
            [
                pl.col("comparison"),
                pl.col("time_point"),
                pl.col("sample_type"),
                pl.col("experiment"),
            ],
            separator="_",
        ).alias("combined_key")
    )
    .rename({"-log10_pval": "neg_log10_pval"})
    .filter(pl.col("comparison").eq("all_conditions_cre+vs.cre-").not_())
    .with_columns(
        pl.lit("").alias("Notes (add initials)"),
        pl.lit("").alias("Figure panels to highlight in (add initials)"),
        pl.col("entry_name").str.split("_").list.get(0).replace(aliases).alias("alias")
    )
)

# Define the known value column names to properly parse the column names
value_cols = ["pep_num", "p_value", "log2_FC", "neg_log10_pval", "Regulation"]

# Get the index columns (these stay in their original order)
index_cols = [
    "entry_name",
    "alias",
    "uniprot_id",
    "uniprot_function",
    "annotation",
    "Mass",
    "gene_name",
    "SWISS_PROT IDs_human_jackson_homology_db",
    "Gene Ontology (GO)",
    "Outcyte",
    "Notes (add initials)",
    "Figure panels to highlight in (add initials)"

]

# Create the pivoted dataframe using the combined key
pivoted_df = df_with_combined.pivot(
    on="combined_key",
    index=index_cols,
    values=value_cols,
    separator="_",
)

# Get all pivoted columns and sort them by experiment AND comparison
pivoted_cols = [col for col in pivoted_df.columns if col not in index_cols]


def parse_column_name(col_name):
    # Find which value column t
    # 
    # 
    # 
    # his starts with
    for val_col in value_cols:
        if col_name.startswith(val_col + "_"):
            # Extract the combined key part (everything after the value column name and separator)
            combined_key = col_name[len(val_col) + 1 :]
            # Split the combined key: comparison_timepoint_sampletype_experiment
            key_parts = combined_key.split("_")
            if len(key_parts) >= 4:
                comparison = "_".join(key_parts[:-1])
                experiment = key_parts[-1]  # Last part is experiment
                return (experiment, comparison)
    return ("", "")  # fallback


pivoted_cols_sorted = sorted(pivoted_cols, key=parse_column_name)

# Reorder the dataframe columns
final_df = pivoted_df.select(index_cols + pivoted_cols_sorted)

# Replace double underscores with single underscores in column names
final_df = final_df.rename({col: col.replace("__", "_") for col in final_df.columns}).with_columns(pl.col("Mass").cast(pl.Int64))

final_df.write_excel(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/03_results/04_turboid_downstream_results/05_results/turboid_differential_expression_wide_format.xlsx",
    freeze_panes = (1,0),
    autofilter = True
)

final_df

In [ ]:
import pandas as pd
data = pl.from_pandas(pd.concat(gsea_results)).select("Term", "experiment", "NES", "comparison")#.pivot(index = ["Term"], values="NES", on = "experiment").drop_nulls()

data.drop_nulls().group_by("Term").agg(
    pl.col("NES").std().alias("std_NES")
).sort(by = "std_NES").tail(10)

pl.from_pandas(pd.concat(gsea_results))

pl.from_pandas(pd.concat(gsea_results)).select(
    "Term", "experiment", "NES", "comparison", "NOM p-val", "FDR q-val"
).filter(pl.col("Term").eq("GOBP_RESPONSE_TO_ENDOPLASMIC_RETICULUM_STRESS"))


# Curating a list of ER stress proteins

In [ ]:
response_to_er_stress = {
    "GOBP_RESPONSE_TO_ENDOPLASMIC_RETICULUM_STRESS": [
        "Abca7",
        "Abl1",
        "Aff4",
        "Afg2b",
        "Agr2",
        "Aifm1",
        "Akt1",
        "Akt2",
        "Akt3",
        "Alox15",
        "Alox5",
        "Amfr",
        "Anks4b",
        "Ankzf1",
        "Apaf1",
        "App",
        "Aqp11",
        "Asb11",
        "Atad3a",
        "Atf3",
        "Atf4",
        "Atf6",
        "Atf6b",
        "Atg10",
        "Atp2a1",
        "Atp2a3",
        "Atxn3",
        "Aup1",
        "Bag6",
        "Bak1",
        "Bax",
        "Bbc3",
        "Bbs1",
        "Bbs10",
        "Bcap31",
        "Bcl2",
        "Bcl2l1",
        "Bcl2l11",
        "Bfar",
        "Bhlha15",
        "Bid",
        "Bok",
        "Brsk2",
        "Calr",
        "Calr3",
        "Calr4",
        "Canx",
        "Casp12",
        "Casp3",
        "Casp9",
        "Cav1",
        "Ccdc47",
        "Ccnd1",
        "Cdk5rap3",
        "Cebpb",
        "Cep290",
        "Cert1",
        "Cftr",
        "Chac1",
        "Clgn",
        "Clu",
        "Cops5",
        "Creb3",
        "Creb3l1",
        "Creb3l2",
        "Creb3l3",
        "Crebrf",
        "D1Pas1",
        "Dab2ip",
        "Ddit3",
        "Ddrgk1",
        "Ddx3x",
        "Derl1",
        "Derl2",
        "Derl3",
        "Dnajb12",
        "Dnajb2",
        "Dnajb9",
        "Dnajc10",
        "Dnajc3",
        "Dnm1l",
        "Ecpas",
        "Edem1",
        "Edem2",
        "Edem3",
        "Eif2a",
        "Eif2ak2",
        "Eif2ak3",
        "Eif2ak4",
        "Eif2b5",
        "Eif2s1",
        "Eif4g1",
        "Elavl4",
        "Erlec1",
        "Erlin1",
        "Erlin2",
        "Ermp1",
        "Ern1",
        "Ern2",
        "Ero1a",
        "Erp27",
        "Erp29",
        "Erp44",
        "Faf1",
        "Faf2",
        "Fbxo17",
        "Fbxo2",
        "Fbxo27",
        "Fbxo44",
        "Fbxo6",
        "Fcgr2b",
        "Ficd",
        "Fis1",
        "Flot1",
        "Foxred2",
        "Get4",
        "Gorasp2",
        "Grina",
        "Gsk3b",
        "H13",
        "Herpud1",
        "Herpud2",
        "Hsp90b1",
        "Hspa5",
        "Hyou1",
        "Ifng",
        "Igtp",
        "Ikbkg",
        "Ins2",
        "Itpr1",
        "Jagn1",
        "Jkamp",
        "Jun",
        "Kcnj8",
        "Lhx1os",
        "Lpcat3",
        "Lrrk2",
        "Man1a",
        "Man1a2",
        "Man1b1",
        "Man1c1",
        "Manf",
        "Map3k20",
        "Map3k5",
        "Marchf6",
        "Marcks",
        "Mbtps2",
        "Nccrp1",
        "Nck1",
        "Nck2",
        "Nfe2l1",
        "Nfe2l2",
        "Nhlrc1",
        "Niban1",
        "Nod1",
        "Nod2",
        "Nploc4",
        "Nr1h2",
        "Nr1h3",
        "Nrbf2",
        "Nrros",
        "Nupr1",
        "Opa1",
        "Os9",
        "P4hb",
        "Park7",
        "Parp16",
        "Parp6",
        "Parp8",
        "Pdia2",
        "Pdia3",
        "Pdia4",
        "Pdia6",
        "Pdx1",
        "Pigbos1",
        "Pik3r1",
        "Pik3r2",
        "Pla2g6",
        "Pmaip1",
        "Pml",
        "Pmp22",
        "Ppp1r15a",
        "Ppp1r15b",
        "Ppp2cb",
        "Prkn",
        "Psmc6",
        "Ptpn1",
        "Ptpn2",
        "Qrich1",
        "Rasgrf1",
        "Rasgrf2",
        "Rcn3",
        "Rhbdd1",
        "Rnf103",
        "Rnf121",
        "Rnf139",
        "Rnf145",
        "Rnf183",
        "Rnf185",
        "Rnf186",
        "Rnf5",
        "Rnft1",
        "Rnft2",
        "Rpap2",
        "Scamp5",
        "Sdf2l1",
        "Sec16a",
        "Sec61b",
        "Sec61bl",
        "Sel1l",
        "Sel1l2",
        "Selenok",
        "Selenon",
        "Selenos",
        "Serinc3",
        "Serp1",
        "Serp2",
        "Sesn2",
        "Sgf29",
        "Sgta",
        "Sirt1",
        "Spop",
        "Srpx",
        "Stc2",
        "Stt3b",
        "Stub1",
        "Svip",
        "Syvn1",
        "Tardbp",
        "Tbl2",
        "Thbs1",
        "Thbs4",
        "Tmbim4",
        "Tmbim6",
        "Tmco1",
        "Tmed2",
        "Tmem117",
        "Tmem129",
        "Tmem238l",
        "Tmem258",
        "Tmem259",
        "Tmem33",
        "Tmem67",
        "Tmtc3",
        "Tmtc4",
        "Tmub1",
        "Tmub2",
        "Tmx1",
        "Tnfrsf10b",
        "Tor1a",
        "Trib3",
        "Trim13",
        "Trim25",
        "Trp53",
        "Txndc12",
        "Uba5",
        "Ubac2",
        "Ube2g2",
        "Ube2j1",
        "Ube2j2",
        "Ube2k",
        "Ube4a",
        "Ube4b",
        "Ubqln1",
        "Ubqln2",
        "Ubxn1",
        "Ubxn10",
        "Ubxn2a",
        "Ubxn4",
        "Ubxn6",
        "Ubxn8",
        "Ufc1",
        "Ufd1",
        "Ufl1",
        "Ufm1",
        "Umod",
        "Usp13",
        "Usp14",
        "Usp19",
        "Usp25",
        "Vapb",
        "Vcp",
        "Wfs1",
        "Xbp1",
        "Yod1",
    ],
    "GOBP_ENDOPLASMIC_RETICULUM_CALCIUM_ION_HOMEOSTASIS":["App","Atp2a1","Atp2a2","Atp2a3","Bak1","Bax","Bcl2","Camk2d","Ccdc47","Clcc1","Dmtn","Grina","Herpud1","Itpr1","Kctd17","Pacs2","Pml","Psen1","Psen2","Rap1gds1","Selenok","Tgm2","Thada","Tmbim6","Tmco1","Tmtc4","Tunar","Wfs1"],
    "GOBP_CALCIUM_ION_HOMEOSTASIS":["Abcc6","Abl1","Adcy8","Adora1","Afg3l2","Akap6","Alpl","Ank","Ank2","Anxa6","Anxa7","Aplnr","Apoe","App","Asph","Atf4","Atg5","Atp13a1","Atp13a2","Atp13a3","Atp13a4","Atp13a5","Atp1b1","Atp2a1","Atp2a2","Atp2a3","Atp2b1","Atp2b2","Atp2b3","Atp2b4","Atp2c1","Atp2c2","Atp6v1b1","Atp7b","Bak1","Bax","Bcl2","Bdkrb1","Bnip3","Bok","Cacna1a","Cacna1c","Cacna1d","Cacna1f","Cacna1s","Cacnb2","Cacnb4","Calb1","Calb2","Calca","Calcb","Calm1","Calm2","Calm3","Camk2d","Capn3","Casq1","Casq2","Casr","Cav1","Cav2","Cav3","Ccdc47","Ccl19","Ccl19-ps1","Ccl19-ps3","Ccl19-ps4","Ccl19-ps5","Ccl19-ps6","Ccl2","Ccl21a","Ccl21b","Ccl21d","Ccl21e","Ccl21f","Ccl3","Ccl5","Ccl8","Ccr1","Ccr1l1","Ccr2","Ccr5","Cd19","Cd40","Cdh23","Cdh5","Cdk5","Cemip","Chd7","Cherp","Chga","Cib2","Cib3","Clcc1","Clec4b1","Cln3","Clstn1","Cnga1","Cngb1","Cnr1","Coro1a","Csrp3","Cst5","Ctrc","Cul5","Cx3cl1","Cxcl10","Cxcl11","Cxcl9","Cxcr3","Cyba","Cyp27b1","Dbi","Ddit3","Dhrs7c","Diaph1","Disc1","Dmd","Dmpk","Dmtn","Drd1","Drd2","Drd3","Drd4","Edn1","Edn2","Edn3","Ednra","Efhc1","Eif3e","Enpp1","Erc1","Erc2","Ero1a","F2","F2r","F2rl3","Fam20a","Fam3a","Fasl","Fate1","Fcrl5","Fgf2","Fgf23","Fgfr1","Fgfr3","Fkbp1a","Fkbp1b","Flna","Frey1","Fto","Fxn","Fxyd1","Fzd9","Gcm2","Ghitm","Glp1r","Gp1ba","Gp1bb","Gp5","Gp9","Gper1","Gpr12","Gpr3","Gpr39","Grid2ip","Grik2","Grin1","Grin2b","Grina","Grm1","Gstm7","Gsto1","Hap1","Hcrtr1","Hcrtr2","Herpud1","Hexb","Hoxa3","Hrc","Htr1b","Htr2a","Htr2b","Htr2c","Htt","Ibtk","Il13","Immt","Inpp4b","Itgav","Itgb3","Itpr1","Itpr2","Itpr3","Jph2","Jph3","Jsrp1","Kcnh1","Kctd17","Kdr","Kel","Kl","Lck","Lcn6","Letm1","Letmd1","Lhcgr","Lime1","Lyn","Maip1","Marcksl1","Mcoln1","Mcu","Mcub","Mcur1","Mettl21c","Mfn2","Micu1","Micu2","Micu3","Ms4a2","Mtln","Myh7b","Myo5a","Ncs1","Ngf","Nol3","Nppc","Npsr1","Nptn","Npy","Npy1r","Nt5e","Ntsr1","Nucb2","Osbpl2","P2rx1","P2rx2","P2rx7","P2ry1","P2ry2","P2ry4","P2ry6","Pacs2","Pde4d","Pdpk1","Pdzd8","Pik3cb","Pkd2","Pkhd1","Plcb1","Plcb2","Plcb3","Plcb4","Plcd1","Plce1","Plcg1","Plcg2","Plch1","Plch2","Plcl1","Plcl2","Pln","Pml","Prkca","Prkcb","Prkce","Prkd1","Psen1","Psen2","Pth","Pth1r","Pthlh","Ptk2b","Ptpn6","Ptprc","Pygm","Rap1gds1","Rgn","Rimbp2","Rmdn3","Ryr1","Ryr2","Ryr3","S100b","Selenok","Selenon","Sgcd","Slc10a7","Slc24a1","Slc24a2","Slc24a3","Slc24a4","Slc24a5","Slc25a23","Slc25a27","Slc30a1","Slc34a1","Slc35g1","Slc37a4","Slc8a1","Slc8a2","Slc8a3","Slc8b1","Smdt1","Snca","Snx10","Spp1","Sppl2c","Sri","Stc1","Stc2","Stim1","Stim2","Stoml2","Sv2a","Sv2b","Synpo","Sypl2","Tcirg1","Tex101","Tgfb1","Tgfb2","Tgm2","Thada","Thy1","Tmbim6","Tmco1","Tmem165","Tmem178","Tmem203","Tmem38a","Tmem38b","Tmem64","Tmtc2","Tmtc4","Tnfsf11","Tnni3","Tpcn2","Trdn","Trim24","Trpa1","Trpc1","Trpc2","Trpc3","Trpc4","Trpc5","Trpc6","Trpc7","Trpm2","Trpm8","Trpv4","Trpv5","Trpv6","Tspoap1","Tunar","Ubash3b","Umod","Vapb","Vdr","Vps54","Wfs1","Wnk4","Wnt5a","Xcl1","Xcr1","Xk","Ywhae","Znhit1"]
}

In [ ]:



aliases = pl.read_csv(
    "/Users/henrysanford/dev/turboid-figures/reference_data/protein_aliases.csv",
    truncate_ragged_lines=True,
)
aliases = dict(zip(aliases["protein"].to_list(), aliases["alias"].to_list()))

fasta_table = (
    pl.read_csv(
        "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/03_results/03_fasta_table_tp_fp/main_fasta_table_without_signal_p.csv",
        ignore_errors=True,
    )
    .filter(
        pl.col("gene_name").is_in(
            response_to_er_stress["GOBP_CALCIUM_ION_HOMEOSTASIS"]
        )
    )
    .with_columns(
        pl.col("entry_name").str.split("_").list.get(0).replace(aliases).alias("alias")
    )
)


list(fasta_table["alias"])